In [ ]:
from pyspark.sql import SparkSession

In [ ]:
spark = (SparkSession.builder
            .appName("shopflow-hello")
            .master("local[*]")
            .getOrCreate())

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("/workspaces/spark")

In [ ]:
raw = spark.read.text (str(PROJECT_ROOT / "data/raw/products.csv"))
raw.show(5, truncate=False)
raw.count()

In [ ]:
products = spark.read.csv(str(PROJECT_ROOT / "data/raw/products.csv"), header=True)

In [ ]:
products.printSchema()

In [ ]:
products.show(5, truncate=False)

definiamo lo schema di orders.jsonl con StructType

In [ ]:
from pyspark.sql.types import (
    StructType,
    StructField, 
    StringType, 
    DoubleType
)

In [ ]:
products_schema = StructType([
    StructField("product_id", StringType(), False),
    StructField("name", StringType(), False),
    StructField("category", StringType(), True),
    StructField("price", DoubleType(), False)
])

leggiamo il JSON con quello schema esplicito

In [ ]:
products = spark.read.schema(products_schema).csv(str(PROJECT_ROOT / "data/raw/products.csv"), header=True)
products.printSchema()
products.show(5, truncate=False)

aggiungiamo una colonna ingestion_ts (timestamp corrente) e una colonna source_file


In [ ]:
from pyspark.sql import functions as F

products_bronze = (
    products
    .withColumn("ingestion_ts", F.current_timestamp())
    .withColumn("source_file", F.input_file_name())
)

In [ ]:
products_bronze.show(5, truncate=False)

scriviamo il risultato come Parquet in data/bronze/customers

In [ ]:
spark.sparkContext.setJobDescription("WRITE bronze products")

applichiamo coalesce(1) per scrivere un solo file Parquet per partizione (utile per piccoli dataset, ma non scalabile per grandi dataset)

In [ ]:
(
    products_bronze
    .coalesce(1)
    .write
    .mode("overwrite")
    .parquet(str(PROJECT_ROOT / "data/bronze/products"))
)